# MtbAIM

Prediction of *Mycobacterium tuberculosis* activity and MIC-family endpoints from SMILES.

**Input**
- Paste SMILES directly. Separate multiple entries with semicolons (`;`).
- Upload a `.txt` file with one SMILES per line.
- Upload a `.csv` file containing a SMILES column.

**Maximum:** 100 compounds per run.


In [ ]:
#@title Predict { display-mode: "form", form-width: "100%" }
#@markdown Select an input method.
input_method = "Paste SMILES"  #@param ["Paste SMILES", "Upload .txt or .csv"]
#@markdown For pasted input, enter one SMILES or separate multiple SMILES with semicolons (`;`).
smiles_input = ""  #@param {type:"string"}
#@markdown Download the complete CSV after prediction.
download_results = True  #@param {type:"boolean"}

import csv
import hashlib
import html
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import textwrap
import urllib.request
import venv
import zipfile
from pathlib import Path

VERSION = "1.3"
SAFE_CWD = Path("/content")
WORK = SAFE_CWD / "MtbAIM"
BUNDLE_DIR = WORK / "bundle"
ENV_DIR = WORK / "env"
INPUT_DIR = WORK / "input"
OUTPUT_DIR = WORK / "results"
WORKER_PATH = WORK / "mtbaim_worker.py"

HF_REPO_ID = "jasdeep002/MtbAIM"
HF_FILENAME = "MTB_AI_predictor_v1.zip"
HF_REVISION = "main"

SAFE_CWD.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_CWD)

WORKER_SOURCE = '#!/usr/bin/env python\nfrom __future__ import annotations\n\nimport argparse\nimport importlib.util\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nEXPECTED_TASKS = {\n    "DS_CLASSIFICATION", "DR_CLASSIFICATION",\n    "DS_MIC_REGRESSION", "DS_MIC90_REGRESSION", "DS_MIC99_REGRESSION",\n    "DR_MIC_REGRESSION", "DR_MIC90_REGRESSION", "DR_MIC99_REGRESSION",\n}\n\ndef load_predictor(root: Path):\n    root = root.resolve()\n    cfg = json.loads((root / "deployment_config.json").read_text(encoding="utf-8"))\n    if int(cfg.get("max_smiles_per_run", -1)) != 100:\n        raise RuntimeError("Unexpected batch-size configuration.")\n\n    engine_path = (root / cfg["inference_engine"]).resolve()\n    spec = importlib.util.spec_from_file_location("mtbaim_engine", engine_path)\n    if spec is None or spec.loader is None:\n        raise RuntimeError("Could not load predictor engine.")\n    engine = importlib.util.module_from_spec(spec)\n    sys.modules["mtbaim_engine"] = engine\n    spec.loader.exec_module(engine)\n\n    # Convert project-relative engine paths to absolute bundle paths.\n    engine.RESULTS_DIR = (root / "results").resolve()\n    engine.RDKIT_DIR = (engine.RESULTS_DIR / "RDKit_morgan").resolve()\n    engine.STEP29_DIR = (engine.RDKIT_DIR / "Step29_final_deployment").resolve()\n    engine.MODEL_DIR = (engine.STEP29_DIR / "models").resolve()\n    engine.OUT_DIR = (engine.RDKIT_DIR / "Step30_predictions").resolve()\n    engine.MASTER_DIR = (engine.RESULTS_DIR / "master_cv_folds").resolve()\n    engine.MANIFEST_PATH = (engine.STEP29_DIR / "29_model_manifest.json").resolve()\n    engine.RECIPE_PATH = (engine.STEP29_DIR / "29_feature_recipe.json").resolve()\n    engine.TRAINING_REFERENCE_PATH = (\n        engine.STEP29_DIR / "29_training_reference_smiles.csv.gz"\n    ).resolve()\n\n    try:\n        from rdkit import RDLogger\n        RDLogger.DisableLog("rdApp.error")\n        RDLogger.DisableLog("rdApp.warning")\n    except Exception:\n        pass\n\n    bundles, feature_columns, _manifest = engine.load_and_validate_step29()\n    if len(feature_columns) != 2265:\n        raise RuntimeError("Unexpected feature count.")\n    if set(bundles) != EXPECTED_TASKS:\n        raise RuntimeError("Predictor task set is incomplete.")\n    return cfg, engine, bundles, feature_columns\n\ndef validate_runtime(root, cfg, engine, bundles, feature_columns):\n    parsed = engine.parse_feature_contract(feature_columns)\n    rd_pos = np.asarray([x[0] for x in parsed[0]], dtype=int)\n    mg_pos = np.asarray([x[0] for x in parsed[1]], dtype=int)\n\n    qc = pd.read_csv(root / cfg["compact_feature_qc"], compression="infer")\n    meta = ["dataset", "source_row_index", "row_id", "canonical_smiles"]\n    if list(qc.columns[:4]) != meta:\n        raise RuntimeError("Feature validation reference is malformed.")\n    if list(map(str, qc.columns[4:])) != list(map(str, feature_columns)):\n        raise RuntimeError("Feature validation schema mismatch.")\n\n    for dataset in ("DS", "DR"):\n        block = qc[qc["dataset"].astype(str) == dataset].reset_index(drop=True)\n        if len(block) == 0:\n            raise RuntimeError("Feature validation reference is incomplete.")\n        x_new = np.empty((len(block), len(feature_columns)), dtype=np.float32)\n        for i, smi in enumerate(block["canonical_smiles"].astype(str)):\n            mol, _canon, _status = engine.canonicalize_smiles(smi)\n            if mol is None:\n                raise RuntimeError("A validation structure could not be parsed.")\n            x_new[i], _ = engine.molecule_feature_vector(\n                mol, feature_columns, parsed\n            )\n        raw = block[feature_columns].apply(\n            pd.to_numeric, errors="coerce"\n        ).to_numpy(dtype=np.float64)\n        x_old, _ = engine.sanitize_numeric_matrix(raw)\n\n        if not np.array_equal(\n            x_new[:, mg_pos], x_old[:, mg_pos], equal_nan=True\n        ):\n            raise RuntimeError("Fingerprint reproduction validation failed.")\n        if not np.array_equal(\n            np.isnan(x_new[:, rd_pos]), np.isnan(x_old[:, rd_pos])\n        ):\n            raise RuntimeError("Descriptor missing-value validation failed.")\n        if not np.allclose(\n            x_new[:, rd_pos], x_old[:, rd_pos],\n            rtol=1e-6, atol=1e-7, equal_nan=True\n        ):\n            raise RuntimeError("Descriptor reproduction validation failed.")\n\n    smoke = pd.read_csv(root / cfg["smoke_prediction_qc"], compression="infer")\n    required = {"task", "canonical_smiles", "expected_step29_prediction"}\n    if not required.issubset(smoke.columns):\n        raise RuntimeError("Prediction validation reference is malformed.")\n\n    cache = {}\n    actual = []\n    labels = []\n    for row in smoke.itertuples(index=False):\n        task = str(row.task)\n        smi = str(row.canonical_smiles)\n        if smi not in cache:\n            mol, _canon, _status = engine.canonicalize_smiles(smi)\n            if mol is None:\n                raise RuntimeError("A prediction-validation structure is invalid.")\n            vec, _ = engine.molecule_feature_vector(\n                mol, feature_columns, parsed\n            )\n            cache[smi] = vec\n\n        b = bundles[task]\n        xt = b["imputer"].transform(\n            cache[smi].reshape(1, -1)\n        ).astype(np.float32, copy=False)\n\n        if task.endswith("_CLASSIFICATION"):\n            classes = list(b["model"].classes_)\n            pos = b.get("positive_class", 1)\n            pred = float(\n                b["model"].predict_proba(xt)[0, classes.index(pos)]\n            )\n            labels.append(\n                "Active" if pred >= float(b["threshold"]) else "Non-active"\n            )\n        else:\n            pred = float(np.asarray(b["model"].predict(xt), dtype=float)[0])\n            labels.append("")\n        actual.append(pred)\n\n    expected = pd.to_numeric(\n        smoke["expected_step29_prediction"], errors="raise"\n    ).to_numpy(dtype=float)\n    if not np.allclose(\n        np.asarray(actual, dtype=float), expected,\n        rtol=1e-6, atol=1e-7,\n    ):\n        raise RuntimeError("Reference prediction validation failed.")\n\n    if "derived_label" in smoke.columns:\n        for i, row in smoke.iterrows():\n            if str(row["task"]).endswith("_CLASSIFICATION"):\n                expected_label = str(row["derived_label"])\n                if expected_label and expected_label.lower() != "nan":\n                    if labels[i] != expected_label:\n                        raise RuntimeError(\n                            "Reference classification validation failed."\n                        )\n\ndef read_input(path: Path, engine):\n    suffix = path.suffix.lower()\n    if suffix == ".txt":\n        lines = [\n            x.strip()\n            for x in path.read_text(encoding="utf-8-sig").splitlines()\n            if x.strip()\n        ]\n        if not lines:\n            raise ValueError("The uploaded text file contains no SMILES.")\n        df = pd.DataFrame({\n            "Compound_ID": [f"Compound_{i+1:03d}" for i in range(len(lines))],\n            "SMILES": lines,\n        })\n        return df, "SMILES"\n\n    if suffix == ".csv":\n        df = pd.read_csv(path)\n        if len(df) == 0:\n            raise ValueError("The uploaded CSV contains no rows.")\n        col = engine.infer_smiles_column(df, None)\n        return df.reset_index(drop=True), col\n\n    raise ValueError("Input must be .txt or .csv.")\n\ndef predict(df, smiles_col, root, engine, bundles, feature_columns):\n    x, canonical, status, query_fps = engine.batch_features(\n        df[smiles_col].tolist(), feature_columns\n    )\n    valid = np.asarray(\n        [str(s).startswith("OK") for s in status], dtype=bool\n    )\n\n    additions = {\n        "MtbAIM_canonical_smiles": canonical,\n        "MtbAIM_smiles_status": status,\n    }\n    additions.update(engine.predict_matrix(x, valid, bundles))\n\n    training_fps = engine.build_training_fps_by_task()\n    for task in sorted(EXPECTED_TASKS):\n        if task not in training_fps:\n            raise RuntimeError("Applicability reference is incomplete.")\n        sims = np.full(len(df), np.nan, dtype=float)\n        for i, bits in enumerate(query_fps):\n            if bits is not None:\n                sims[i] = engine.nearest_training_similarity(\n                    bits, training_fps[task]\n                )\n        additions[f"{task}_max_train_tanimoto"] = sims\n\n    return pd.concat(\n        [df.reset_index(drop=True), pd.DataFrame(additions)],\n        axis=1,\n        copy=False,\n    )\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--bundle-root", required=True)\n    p.add_argument("--input", default=None)\n    p.add_argument("--output", default=None)\n    p.add_argument("--self-check", action="store_true")\n    args = p.parse_args()\n\n    root = Path(args.bundle_root).resolve()\n    cfg, engine, bundles, feature_columns = load_predictor(root)\n    validate_runtime(root, cfg, engine, bundles, feature_columns)\n\n    if args.self_check:\n        print("PREDICTOR_CHECK_PASS")\n        return\n\n    if not args.input or not args.output:\n        raise RuntimeError("--input and --output are required for prediction.")\n\n    input_path = Path(args.input).resolve()\n    df, smiles_col = read_input(input_path, engine)\n\n    if len(df) < 1:\n        raise ValueError("No compounds were supplied.")\n    if len(df) > 100:\n        raise ValueError(\n            f"A maximum of 100 compounds can be submitted per run. Received {len(df)}."\n        )\n\n    result = predict(\n        df, smiles_col, root, engine, bundles, feature_columns\n    )\n    out = Path(args.output).resolve()\n    out.parent.mkdir(parents=True, exist_ok=True)\n    result.to_csv(out, index=False)\n\n    n_valid = int(\n        result["MtbAIM_smiles_status"].astype(str).str.startswith("OK").sum()\n    )\n    n_invalid = int(len(result) - n_valid)\n    print(\n        f"PREDICTION_PASS rows={len(result)} valid={n_valid} invalid={n_invalid}"\n    )\n\nif __name__ == "__main__":\n    main()\n'

def _sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _run(cmd, cwd=SAFE_CWD, timeout=1800):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(Path(cwd).resolve()),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        timeout=timeout,
    )
    if p.returncode != 0:
        detail = (p.stderr or p.stdout or "")[-5000:]
        raise RuntimeError("Predictor setup failed.\n" + detail)
    return p

def _safe_extract(zip_path, destination):
    root = destination.resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        bad = zf.testzip()
        if bad is not None:
            raise RuntimeError("The predictor archive failed its integrity check.")
        infos = zf.infolist()
        if len(infos) > 200:
            raise RuntimeError("Unexpected predictor archive structure.")
        if sum(int(x.file_size) for x in infos) > 4 * 1024**3:
            raise RuntimeError("The predictor archive is unexpectedly large.")
        for info in infos:
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise RuntimeError("Unsupported link in predictor archive.")
            target = (destination / info.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError("Unsafe path in predictor archive.")
        zf.extractall(destination)

def _download_bundle():
    WORK.mkdir(parents=True, exist_ok=True)
    archive = WORK / HF_FILENAME
    url = (
        "https://huggingface.co/datasets/"
        + HF_REPO_ID
        + "/resolve/"
        + HF_REVISION
        + "/"
        + HF_FILENAME
        + "?download=true"
    )
    req = urllib.request.Request(
        url, headers={"User-Agent": "MtbAIM-Colab/1.3"}
    )
    with urllib.request.urlopen(req, timeout=300) as response:
        with archive.open("wb") as fh:
            shutil.copyfileobj(response, fh)
    if not archive.exists() or archive.stat().st_size == 0:
        raise RuntimeError("The predictor archive could not be downloaded.")
    return archive

def _prepare_bundle():
    archive = _download_bundle()

    if BUNDLE_DIR.exists():
        shutil.rmtree(BUNDLE_DIR)
    BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
    _safe_extract(archive, BUNDLE_DIR)

    configs = list(BUNDLE_DIR.rglob("deployment_config.json"))
    if len(configs) != 1:
        raise RuntimeError("Unexpected predictor archive structure.")
    root = configs[0].parent.resolve()

    validator = root / "validate_bundle.py"
    if not validator.exists():
        raise RuntimeError("Predictor validation information is missing.")
    check = _run([sys.executable, validator], cwd=SAFE_CWD, timeout=120)
    if "BUNDLE VALIDATION PASS" not in check.stdout:
        raise RuntimeError("Predictor archive validation failed.")
    return root

def _filtered_requirements(root):
    source = root / "requirements_colab.txt"
    if not source.exists():
        raise RuntimeError("Predictor requirements are missing.")

    allowed = {
        "numpy", "pandas", "scikit-learn", "joblib",
        "lightgbm", "xgboost", "rdkit",
    }
    kept = []
    for raw in source.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        name = re.split(r"[<>=!~\[]", line, maxsplit=1)[0].strip().lower()
        if name in allowed:
            kept.append(line)

    required_names = {
        re.split(r"[<>=!~\[]", x, maxsplit=1)[0].strip().lower()
        for x in kept
    }
    if not {"numpy", "pandas", "scikit-learn", "joblib", "rdkit"}.issubset(required_names):
        raise RuntimeError("Predictor requirements are incomplete.")

    path = WORK / "requirements_inference.txt"
    path.write_text("\n".join(kept) + "\n", encoding="utf-8")
    return path

def _prepare_environment(root):
    req = _filtered_requirements(root)
    req_hash = _sha256(req)
    marker = ENV_DIR / ".mtbaim_requirements_sha256"
    env_python = (
        ENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    )

    reuse = (
        env_python.exists()
        and marker.exists()
        and marker.read_text(encoding="utf-8").strip() == req_hash
    )

    if not reuse:
        if ENV_DIR.exists():
            shutil.rmtree(ENV_DIR)
        venv.EnvBuilder(with_pip=True, clear=True).create(ENV_DIR)
        env_python = ENV_DIR / "bin" / "python"
        _run(
            [
                env_python, "-m", "pip", "install",
                "--quiet", "--disable-pip-version-check",
                "-r", req,
            ],
            cwd=SAFE_CWD,
            timeout=1800,
        )
        marker.write_text(req_hash, encoding="utf-8")

    WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")

    check = _run(
        [
            env_python, WORKER_PATH,
            "--bundle-root", root,
            "--self-check",
        ],
        cwd=SAFE_CWD,
        timeout=600,
    )
    if "PREDICTOR_CHECK_PASS" not in check.stdout:
        raise RuntimeError("Predictor validation failed.")
    return env_python

def _parse_pasted(text):
    items = []
    for line in str(text or "").replace("\r", "\n").split("\n"):
        for item in line.split(";"):
            item = item.strip()
            if item:
                items.append(item)
    if not items:
        raise ValueError("Enter at least one SMILES.")
    if len(items) > 100:
        raise ValueError(
            f"A maximum of 100 compounds can be submitted per run. Received {len(items)}."
        )
    return items

def _prepare_input():
    INPUT_DIR.mkdir(parents=True, exist_ok=True)

    if input_method == "Paste SMILES":
        items = _parse_pasted(smiles_input)
        path = INPUT_DIR / "submitted_smiles.csv"
        with path.open("w", newline="", encoding="utf-8") as fh:
            w = csv.writer(fh)
            w.writerow(["Compound_ID", "SMILES"])
            for i, smi in enumerate(items, start=1):
                w.writerow([f"Compound_{i:03d}", smi])
        return path

    from google.colab import files
    print("Upload one .txt or .csv file.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one .txt or .csv file.")
    name = next(iter(uploaded))
    suffix = Path(name).suffix.lower()
    if suffix not in {".txt", ".csv"}:
        raise ValueError("Upload a .txt or .csv file.")
    path = INPUT_DIR / ("submitted" + suffix)
    path.write_bytes(uploaded[name])
    return path

def _render_preview(csv_path):
    from IPython.display import HTML, display

    wanted = [
        ("Compound_ID", "Compound"),
        ("SMILES", "SMILES"),
        ("MtbAIM_smiles_status", "Status"),
        ("DS_activity_call", "DS activity"),
        ("DS_probability_active", "DS P(active)"),
        ("DR_activity_call", "DR activity"),
        ("DR_probability_active", "DR P(active)"),
        ("DS_MIC_uM", "DS MIC (µM)"),
        ("DS_MIC90_uM", "DS MIC90 (µM)"),
        ("DS_MIC99_uM", "DS MIC99 (µM)"),
        ("DR_MIC_uM", "DR MIC (µM)"),
        ("DR_MIC90_uM", "DR MIC90 (µM)"),
        ("DR_MIC99_uM", "DR MIC99 (µM)"),
    ]

    with Path(csv_path).open("r", encoding="utf-8-sig", newline="") as fh:
        rows = list(csv.DictReader(fh))

    cols = [(src, title) for src, title in wanted if rows and src in rows[0]]
    if not rows:
        print("No prediction rows returned.")
        return

    def fmt(src, value):
        if value in (None, "", "nan", "NaN"):
            return ""
        if src.endswith("_probability_active"):
            try:
                return f"{float(value):.4f}"
            except Exception:
                return str(value)
        if src.endswith("_uM"):
            try:
                return f"{float(value):.4g}"
            except Exception:
                return str(value)
        return str(value)

    parts = [
        '<div style="overflow-x:auto">',
        '<table style="border-collapse:collapse;font-size:13px">',
        "<thead><tr>",
    ]
    for _src, title in cols:
        parts.append(
            '<th style="border:1px solid #ddd;padding:6px 8px;white-space:nowrap">'
            + html.escape(title) + "</th>"
        )
    parts.append("</tr></thead><tbody>")
    for row in rows:
        parts.append("<tr>")
        for src, _title in cols:
            parts.append(
                '<td style="border:1px solid #ddd;padding:6px 8px;white-space:nowrap">'
                + html.escape(fmt(src, row.get(src, ""))) + "</td>"
            )
        parts.append("</tr>")
    parts.append("</tbody></table></div>")
    display(HTML("".join(parts)))

print(f"Preparing predictor (version {VERSION})...")

root = _prepare_bundle()
env_python = _prepare_environment(root)
input_path = _prepare_input()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / "MtbAIM_predictions.csv"

run = _run(
    [
        env_python, WORKER_PATH,
        "--bundle-root", root,
        "--input", input_path,
        "--output", output_path,
    ],
    cwd=SAFE_CWD,
    timeout=900,
)

if "PREDICTION_PASS" not in run.stdout:
    raise RuntimeError("Prediction did not complete.")

_render_preview(output_path)
print("\nPrediction complete.")
print("Results: MtbAIM_predictions.csv")

if download_results:
    from google.colab import files
    files.download(str(output_path))


### Output

The table reports DS and DR activity predictions together with MIC, MIC90, and MIC99 estimates. The downloaded CSV contains the complete prediction output.
